# 11 — Grouped Feature Ablation

Compare a predefined all-feature model, feature-family models, and leave-one-feature-out variants using the same host-star-grouped folds.

**Reproducibility note:** This notebook follows Notebook 03's primary sample definition and Random Forest pipeline. It does not overwrite or modify any earlier notebook. Run cells from top to bottom in Kaggle or Jupyter.

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, precision_score, recall_score, f1_score, log_loss
)

# Match Notebook 03's primary population and modeling settings.
RADIUS_MIN, RADIUS_MAX = 1.0, 4.0
TEFF_MIN, TEFF_MAX = 4700, 6500
N_SPLITS = 5
THRESHOLD = 0.50
RANDOM_STATE = 42
N_ESTIMATORS = 400

# Set this manually if auto-detection finds zero or multiple candidate CSV files.
DATA_PATH = None  # e.g. "/kaggle/input/your-dataset/cumulative.csv"

def locate_koi_csv():
    if DATA_PATH is not None:
        if not os.path.exists(DATA_PATH):
            raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")
        return DATA_PATH
    candidates = sorted(
        p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
        if ("cumulative" in os.path.basename(p).lower()
            or "koi" in os.path.basename(p).lower())
    )
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "No KOI cumulative CSV found under /kaggle/input. "
            "Attach the same catalog used in Notebook 03 and set DATA_PATH manually."
        )
    print("Possible KOI CSV files:")
    for p in candidates:
        print(" -", p)
    raise ValueError("Multiple possible KOI CSVs found. Set DATA_PATH explicitly.")

def load_main_sample():
    path = locate_koi_csv()
    print("Using catalog:", path)
    raw = pd.read_csv(path, comment="#", low_memory=False)
    required = {"kepid", "koi_disposition", "koi_prad", "koi_steff"}
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"Required columns missing: {sorted(missing)}")
    for col in ["kepid", "koi_prad", "koi_steff"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    catalog = raw[
        raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"])
        & raw["kepid"].notna()
        & raw["koi_prad"].between(RADIUS_MIN, RADIUS_MAX, inclusive="both")
        & raw["koi_steff"].between(TEFF_MIN, TEFF_MAX, inclusive="both")
    ].copy()
    catalog["group"] = catalog["kepid"].astype("int64")
    labeled = catalog[
        catalog["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
    ].copy()
    labeled["label"] = (labeled["koi_disposition"] == "CONFIRMED").astype(int)
    candidate_features = [
        "koi_period", "koi_time0bk", "koi_impact", "koi_duration",
        "koi_depth", "koi_prad", "koi_teq", "koi_insol",
        "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"
    ]
    features = [c for c in candidate_features if c in labeled.columns]
    if len(features) < 3:
        raise ValueError(f"Fewer than 3 Notebook 03 features available: {features}")
    X = labeled[features].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    drop = [c for c in X if X[c].notna().sum() == 0 or X[c].nunique(dropna=True) <= 1]
    if drop:
        print("Dropping all-missing/constant features:", drop)
        X = X.drop(columns=drop)
    y = labeled["label"].to_numpy(dtype=int)
    groups = labeled["group"].to_numpy()
    if len(np.unique(y)) != 2:
        raise ValueError("Both CONFIRMED and FALSE POSITIVE labels are required.")
    if labeled.groupby("label")["group"].nunique().min() < N_SPLITS:
        raise ValueError("Too few distinct host stars in at least one class for 5 folds.")
    print(f"Filtered catalog rows: {len(catalog):,}")
    print(f"Resolved labeled rows: {len(labeled):,}")
    print(f"Unique resolved host stars: {len(np.unique(groups)):,}")
    print("Class counts (0=false positive, 1=confirmed):")
    print(pd.Series(y).value_counts().sort_index())
    print("Features:", list(X.columns))
    return path, raw, catalog, labeled, X, y, groups

def build_model(seed=RANDOM_STATE):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        ))
    ])

def make_oof(X, y, groups, grouped=True, seed=RANDOM_STATE, n_splits=N_SPLITS):
    splitter = (
        StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        if grouped else
        StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    )
    probs = np.full(len(y), np.nan, dtype=float)
    folds = np.full(len(y), -1, dtype=int)
    split_iter = splitter.split(X, y, groups) if grouped else splitter.split(X, y)
    for fold, (tr, te) in enumerate(split_iter):
        if np.unique(y[tr]).size != 2 or np.unique(y[te]).size != 2:
            raise ValueError(
                f"Fold {fold+1} has a single class. Inspect group distribution; "
                "do not silently report fold-level class metrics."
            )
        if grouped and set(groups[tr]).intersection(set(groups[te])):
            raise RuntimeError(f"Host-star overlap in fold {fold+1}.")
        model = build_model(seed=seed)
        model.fit(X.iloc[tr], y[tr])
        probs[te] = model.predict_proba(X.iloc[te])[:, 1]
        folds[te] = fold
    if np.isnan(probs).any() or (folds < 0).any():
        raise RuntimeError("Every row must receive exactly one out-of-fold prediction.")
    return probs, folds

def metric_dict(y, p, threshold=THRESHOLD):
    pred = (p >= threshold).astype(int)
    return {
        "ROC_AUC": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "Brier_score_lower_is_better": brier_score_loss(y, p),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "threshold": threshold
    }

def save_csv(df, filename):
    os.makedirs("/kaggle/working", exist_ok=True)
    path = os.path.join("/kaggle/working", filename)
    df.to_csv(path, index=False)
    print("Saved:", path)

## Aim
Test which approved physical/observational feature groups contribute to held-out classification performance under host-star-grouped validation. Feature groups are predefined from Notebook 03's feature allow-list. All preprocessing remains inside each training fold.

In [2]:
path, raw, catalog, labeled, X, y, groups = load_main_sample()

FEATURE_GROUPS = {
    "stellar_properties": ["koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"],
    "planet_properties": ["koi_prad", "koi_teq", "koi_insol"],
    "transit_observations": [
        "koi_period", "koi_time0bk", "koi_impact", "koi_duration", "koi_depth"
    ]
}
available = set(X.columns)
experiments = {"all_features": list(X.columns)}
for name, proposed in FEATURE_GROUPS.items():
    cols = [c for c in proposed if c in available]
    if len(cols) >= 2:
        experiments[name] = cols
    else:
        print(f"Skipping {name}: fewer than two available features ({cols})")

# Leave-one-feature-out tests show whether a single predictor materially changes scores.
for feature in X.columns:
    remaining = [c for c in X.columns if c != feature]
    if len(remaining) >= 2:
        experiments[f"without_{feature}"] = remaining

results = []
predictions = []
for name, cols in experiments.items():
    print(f"\n=== {name}: {len(cols)} features ===")
    probs, fold_ids = make_oof(X[cols], y, groups, grouped=True, seed=RANDOM_STATE)
    results.append({"experiment": name, "n_features": len(cols), **metric_dict(y, probs)})
    predictions.append(pd.DataFrame({
        "experiment": name, "row_index": labeled.index,
        "host_id": groups, "true_label": y,
        "grouped_oof_probability": probs, "fold": fold_ids
    }))

ablation = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
display(ablation.round(4))
save_csv(ablation, "11_grouped_feature_ablation_metrics.csv")
save_csv(pd.concat(predictions, ignore_index=True), "11_grouped_feature_ablation_oof.csv")

Using catalog: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv
Filtered catalog rows: 4,249
Resolved labeled rows: 3,195
Unique resolved host stars: 2,654
Class counts (0=false positive, 1=confirmed):
0    1229
1    1966
Name: count, dtype: int64
Features: ['koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag']

=== all_features: 12 features ===

=== stellar_properties: 4 features ===

=== planet_properties: 3 features ===

=== transit_observations: 5 features ===

=== without_koi_period: 11 features ===

=== without_koi_time0bk: 11 features ===

=== without_koi_impact: 11 features ===

=== without_koi_duration: 11 features ===

=== without_koi_depth: 11 features ===

=== without_koi_prad: 11 features ===

=== without_koi_teq: 11 features ===

=== without_koi_insol: 11 features ===

=== without_koi_steff: 11 features ===

=== without_koi_slogg:

,experiment,n_features,ROC_AUC,average_precision,balanced_accuracy,Brier_score_lower_is_better,precision,recall,F1,threshold
5,without_koi_time0bk,11,0.9249,0.9442,0.8480,0.1042,0.8702,0.9140,0.8916,0.5
11,without_koi_insol,11,0.9247,0.9436,0.8469,0.1048,0.8693,0.9135,0.8909,0.5
10,without_koi_teq,11,0.9243,0.9425,0.8486,0.1051,0.8707,0.9145,0.8921,0.5
8,without_koi_depth,11,0.9241,0.9419,0.8437,0.1047,0.8644,0.9176,0.8902,0.5
0,all_features,12,0.9236,0.9438,0.8450,0.1057,0.8676,0.9130,0.8897,0.5
9,without_koi_prad,11,0.9234,0.9437,0.8419,0.1057,0.8630,0.9166,0.8890,0.5
13,without_koi_slogg,11,0.9231,0.9439,0.8448,0.1055,0.8666,0.9151,0.8902,0.5
14,without_koi_srad,11,0.9229,0.9431,0.8426,0.1057,0.8651,0.9130,0.8884,0.5
4,without_koi_period,11,0.9212,0.9426,0.8384,0.1079,0.8627,0.9079,0.8848,0.5
15,without_koi_kepmag,11,0.9203,0.9405,0.8372,0.1077,0.8600,0.9120,0.8852,0.5


Saved: /kaggle/working/11_grouped_feature_ablation_metrics.csv
Saved: /kaggle/working/11_grouped_feature_ablation_oof.csv


## Interpretation guardrails
- This is an ablation/sensitivity analysis, not a causal estimate of a feature's importance.
- Feature groups overlap conceptually only by their stated definitions; the categories are analytical choices.
- The `without_...` experiments are evaluated one at a time against the all-feature model.
- A feature can be predictive because of catalog selection or measurement practices; interpret its role scientifically before making an astrophysical claim.